In [ ]:
import pandas as pd
pd.plotting.register_matplotlib_converters()
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
print("Setup Complete")

In [1]:
!pip install gdown
!gdown --fuzzy https://drive.google.com/open?id=1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj -O /kaggle/working/Celeb-DF-v2.zip

Downloading...
From (original): https://drive.google.com/uc?id=1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj
From (redirected): https://drive.google.com/uc?id=1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj&confirm=t&uuid=84dc388c-58f4-4fd1-8587-c712811a8944
To: /kaggle/working/Celeb-DF-v2.zip
100%|███████████████████████████████████████| 9.95G/9.95G [00:58<00:00, 171MB/s]


In [3]:
!unzip -q /kaggle/working/Celeb-DF-v2.zip -d /kaggle/working/celebdf

replace /kaggle/working/celebdf/Celeb-real/id0_0000.mp4? [y]es, [n]o, [A]ll, [N]one, [r]ename: ^C


In [6]:
!rm -rf /kaggle/working/celebdf

In [7]:
!unzip -q -o /kaggle/working/Celeb-DF-v2.zip -d /kaggle/working/celebdf

In [8]:
!rm -rf /kaggle/working/celebdf

In [9]:
import zipfile
z = zipfile.ZipFile('/kaggle/working/Celeb-DF-v2.zip')
names = z.namelist()
print(len(names), "total files")
for prefix in ['Celeb-real/', 'Celeb-synthesis/', 'YouTube-real/']:
    matches = [n for n in names if n.startswith(prefix)]
    print(prefix, len(matches), 'files, e.g.', matches[:3])

6533 total files
Celeb-real/ 591 files, e.g. ['Celeb-real/', 'Celeb-real/id0_0000.mp4', 'Celeb-real/id0_0001.mp4']
Celeb-synthesis/ 5640 files, e.g. ['Celeb-synthesis/', 'Celeb-synthesis/id0_id16_0000.mp4', 'Celeb-synthesis/id0_id16_0001.mp4']
YouTube-real/ 301 files, e.g. ['YouTube-real/', 'YouTube-real/00000.mp4', 'YouTube-real/00001.mp4']


In [12]:
import re
from collections import defaultdict

# Parse identity numbers from filenames
real_videos = [n for n in names if n.startswith('Celeb-real/') and n.endswith('.mp4')]
fake_videos = [n for n in names if n.startswith('Celeb-synthesis/') and n.endswith('.mp4')]
yt_real_videos = [n for n in names if n.startswith('YouTube-real/') and n.endswith('.mp4')]

def real_id(fname):
    m = re.search(r'id(\d+)_\d+\.mp4$', fname)
    return int(m.group(1)) if m else None

def fake_ids(fname):
    m = re.search(r'id(\d+)_id(\d+)_\d+\.mp4$', fname)
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)

all_identities = sorted(set(real_id(f) for f in real_videos))
print(f"Total identities in Celeb-real: {len(all_identities)}")
print(f"Total real videos: {len(real_videos)}, fake videos: {len(fake_videos)}, YouTube-real: {len(yt_real_videos)}")

# 80/20 identity-level split (seed 42 for reproducibility, matching your existing convention)
import random
random.seed(42)
shuffled_ids = all_identities.copy()
random.shuffle(shuffled_ids)
split_point = int(len(shuffled_ids) * 0.8)
train_ids = set(shuffled_ids[:split_point])
val_ids = set(shuffled_ids[split_point:])
print(f"Train identities: {len(train_ids)}, Val identities: {len(val_ids)}")

# Assign real videos by identity
train_real = [f for f in real_videos if real_id(f) in train_ids]
val_real   = [f for f in real_videos if real_id(f) in val_ids]

# Assign fake videos: goes to val if EITHER source or target identity is a val identity
train_fake, val_fake = [], []
for f in fake_videos:
    src, tgt = fake_ids(f)
    if src in val_ids or tgt in val_ids:
        val_fake.append(f)
    else:
        train_fake.append(f)

print(f"\nTrain: {len(train_real)} real, {len(train_fake)} fake")
print(f"Val:   {len(val_real)} real, {len(val_fake)} fake")

Total identities in Celeb-real: 59
Total real videos: 590, fake videos: 5639, YouTube-real: 300
Train identities: 47, Val identities: 12

Train: 466 real, 3427 fake
Val:   124 real, 2212 fake


In [13]:
test_list_files = [n for n in names if 'test' in n.lower() or 'List_of' in n]
print(test_list_files)

if test_list_files:
    with z.open(test_list_files[0]) as f:
        content = f.read().decode('utf-8')
    lines = content.strip().split('\n')
    print(f"\n{len(lines)} lines, first 5:")
    for l in lines[:5]:
        print(repr(l))

['List_of_testing_videos.txt']

518 lines, first 5:
'1 YouTube-real/00170.mp4'
'1 YouTube-real/00208.mp4'
'1 YouTube-real/00063.mp4'
'1 YouTube-real/00024.mp4'
'1 YouTube-real/00021.mp4'


In [14]:
test_set = set()
for line in lines:
    parts = line.strip().split()
    label, path = int(parts[0]), parts[1]
    test_set.add(path)

print(f"{len(test_set)} unique test videos")

all_videos = real_videos + fake_videos + yt_real_videos
train_videos = [v for v in all_videos if v not in test_set]
test_videos  = [v for v in all_videos if v in test_set]

test_real = [v for v in test_videos if 'synthesis' not in v]
test_fake = [v for v in test_videos if 'synthesis' in v]

print(f"Train: {len(train_videos)} videos")
print(f"Test:  {len(test_videos)} videos  ({len(test_real)} real, {len(test_fake)} fake)")

518 unique test videos
Train: 6011 videos
Test:  518 videos  (178 real, 340 fake)


In [15]:
train_ids = set(real_id(v) for v in train_videos if v.startswith('Celeb-real/'))
test_ids  = set(real_id(v) for v in test_videos if v.startswith('Celeb-real/') or 'synthesis' in v and real_id(v))
overlap = train_ids & test_ids
print(f"Train identities: {len(train_ids)}, Test identities: {len(test_ids)}, Overlap: {len(overlap)}")

Train identities: 59, Test identities: 56, Overlap: 56


In [ ]:
import zipfile, cv2, numpy as np, os, random, re, tempfile

random.seed(42)
ZIP_PATH = '/kaggle/working/Celeb-DF-v2.zip'
OUT_DIR = '/kaggle/working/frames'
FRAMES_PER_VIDEO_TRAIN = 15
FRAMES_PER_VIDEO_TEST = 20
TRAIN_VIDEO_CAP = 2000

for split in ['train', 'test']:
    for label in ['real', 'fake']:
        os.makedirs(f'{OUT_DIR}/{split}/{label}', exist_ok=True)

z = zipfile.ZipFile(ZIP_PATH)
names = z.namelist()
real_videos = [n for n in names if n.startswith('Celeb-real/') and n.endswith('.mp4')]
fake_videos = [n for n in names if n.startswith('Celeb-synthesis/') and n.endswith('.mp4')]

with z.open('List_of_testing_videos.txt') as f:
    test_lines = f.read().decode('utf-8').strip().split('\n')
test_set = set(line.strip().split()[1] for line in test_lines)

all_videos = [(v, 'real') for v in real_videos] + [(v, 'fake') for v in fake_videos]
train_pool = [(v, l) for v, l in all_videos if v not in test_set]
test_pool  = [(v, l) for v, l in all_videos if v in test_set]

train_real = [x for x in train_pool if x[1] == 'real']; random.shuffle(train_real)
train_fake = [x for x in train_pool if x[1] == 'fake']; random.shuffle(train_fake)
n_real = min(len(train_real), TRAIN_VIDEO_CAP // 3)
n_fake = min(len(train_fake), TRAIN_VIDEO_CAP - n_real)
train_selected = train_real[:n_real] + train_fake[:n_fake]

print(f"Train: {len(train_selected)} videos ({n_real} real, {n_fake} fake)")
print(f"Test:  {len(test_pool)} videos (official, full)")

def extract_frames(entry, n_frames, split):
    video_path, label = entry
    safe_name = video_path.replace('/', '__').replace('.mp4', '')
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
        tmp.write(z.read(video_path)); tmp_path = tmp.name
    saved = 0
    try:
        cap = cv2.VideoCapture(tmp_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total > 0:
            idxs = np.linspace(0, total - 1, min(n_frames, total)).astype(int)
            for i, idx in enumerate(idxs):
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ok, frame = cap.read()
                if ok:
                    cv2.imwrite(f'{OUT_DIR}/{split}/{label}/{safe_name}_{i:03d}.jpg', frame)
                    saved += 1
        cap.release()
    finally:
        os.remove(tmp_path)
    return saved

total = 0
for i, entry in enumerate(train_selected):
    total += extract_frames(entry, FRAMES_PER_VIDEO_TRAIN, 'train')
    if i % 200 == 0: print(f"train {i}/{len(train_selected)}, frames so far: {total}")

for i, entry in enumerate(test_pool):
    total += extract_frames(entry, FRAMES_PER_VIDEO_TEST, 'test')
    if i % 100 == 0: print(f"test {i}/{len(test_pool)}, frames so far: {total}")

print(f"\nDone. Total frames: {total}")
for split in ['train', 'test']:
    for label in ['real', 'fake']:
        print(split, label, len(os.listdir(f'{OUT_DIR}/{split}/{label}')))

Train: 2000 videos (482 real, 1518 fake)
Test:  448 videos (official, full)
train 0/2000, frames so far: 15
train 200/2000, frames so far: 3001
train 400/2000, frames so far: 6001
train 600/2000, frames so far: 9001
train 800/2000, frames so far: 12001
train 1000/2000, frames so far: 15001
train 1200/2000, frames so far: 18001
train 1400/2000, frames so far: 21001
train 1600/2000, frames so far: 24001
train 1800/2000, frames so far: 27001
test 0/448, frames so far: 30006
test 100/448, frames so far: 32006
test 200/448, frames so far: 34006
test 300/448, frames so far: 36006


In [17]:
import glob
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T

class CelebDFFrameDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        self.transform = transform
        real = glob.glob(os.path.join(root_dir, split, 'real', '*.jpg'))
        fake = glob.glob(os.path.join(root_dir, split, 'fake', '*.jpg'))
        self.samples = [(p, 0) for p in real] + [(p, 1) for p in fake]
        print(f"{split}: {len(real)} real, {len(fake)} fake")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label, path

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
model_transform = T.Compose([T.Resize((224,224)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
raw_transform   = T.Compose([T.Resize((224,224)), T.ToTensor()])

train_ds_model = CelebDFFrameDataset('/kaggle/working/frames', 'train', model_transform)
test_ds_model  = CelebDFFrameDataset('/kaggle/working/frames', 'test', model_transform)
train_ds_raw   = CelebDFFrameDataset('/kaggle/working/frames', 'train', raw_transform)
test_ds_raw    = CelebDFFrameDataset('/kaggle/working/frames', 'test', raw_transform)

train: 7216 real, 22770 fake
test: 2160 real, 6800 fake
train: 7216 real, 22770 fake
test: 2160 real, 6800 fake


In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

def compute_hfr(img_tensor, radius_frac=0.3):
    gray = (0.299*img_tensor[0] + 0.587*img_tensor[1] + 0.114*img_tensor[2]).numpy()
    fshift = np.fft.fftshift(np.fft.fft2(gray))
    mag = np.abs(fshift)
    h, w = mag.shape
    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((X-w//2)**2 + (Y-h//2)**2)
    radius = radius_frac * min(h,w) / 2
    return mag[dist > radius].sum() / (mag.sum() + 1e-8)

def fft_scores(dataset):
    hfrs, labels = [], []
    for i in range(len(dataset)):
        img, label, _ = dataset[i]
        hfrs.append(compute_hfr(img)); labels.append(label)
    return np.array(hfrs), np.array(labels)

train_hfr, train_labels = fft_scores(train_ds_raw)
best_thresh, best_f1 = None, -1
for t in np.linspace(train_hfr.min(), train_hfr.max(), 50):
    f1 = f1_score(train_labels, (train_hfr > t).astype(int))
    if f1 > best_f1: best_f1, best_thresh = f1, t
print(f"Calibrated threshold: {best_thresh:.4f} (train F1={best_f1:.4f})")

test_hfr, test_labels = fft_scores(test_ds_raw)
test_pred_fft = (test_hfr > best_thresh).astype(int)
test_conf_fft = 1 / (1 + np.exp(-(test_hfr - best_thresh) * 10))

def compute_metrics(y_true, y_pred, y_score):
    return {'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred),
            'recall': recall_score(y_true, y_pred), 'f1': f1_score(y_true, y_pred),
            'auc_roc': roc_auc_score(y_true, y_score)}

fft_metrics = compute_metrics(test_labels, test_pred_fft, test_conf_fft)
print("FFT Detector on official test set:", fft_metrics)

Calibrated threshold: 0.0000 (train F1=0.8632)
FFT Detector on official test set: {'accuracy': 0.7589285714285714, 'precision': 0.7589285714285714, 'recall': 1.0, 'f1': 0.8629441624365483, 'auc_roc': np.float64(0.492490366285403)}


In [19]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.classifier[1].in_features, 1))
model = model.to(device)

train_loader = DataLoader(train_ds_model, batch_size=32, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_ds_model, batch_size=32, shuffle=False, num_workers=2)
criterion = nn.BCEWithLogitsLoss()

def get_optimizer(phase):
    for p in model.features.parameters(): p.requires_grad = (phase == 2)
    if phase == 1:
        return torch.optim.AdamW(model.classifier.parameters(), lr=1e-4, weight_decay=1e-4)
    return torch.optim.AdamW([
        {'params': model.classifier.parameters(), 'lr': 1e-4},
        {'params': model.features.parameters(), 'lr': 1e-5}], weight_decay=1e-4)

def evaluate_model(loader):
    model.eval(); labels_all, scores_all = [], []
    with torch.no_grad():
        for imgs, labels, _ in loader:
            probs = torch.sigmoid(model(imgs.to(device)).squeeze(1)).cpu().numpy()
            scores_all.extend(probs); labels_all.extend(labels.numpy())
    return np.array(labels_all), np.array(scores_all)

best_auc, history = -1, []
EPOCHS_P1, EPOCHS_P2 = 3, 7

for epoch in range(1, EPOCHS_P1 + EPOCHS_P2 + 1):
    phase = 1 if epoch <= EPOCHS_P1 else 2
    if epoch in (1, EPOCHS_P1 + 1):
        optimizer = get_optimizer(phase)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=(EPOCHS_P1 if phase==1 else EPOCHS_P2))
    model.train(); running_loss = 0
    for imgs, labels, _ in train_loader:
        imgs, labels = imgs.to(device), labels.float().to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs).squeeze(1), labels)
        loss.backward(); optimizer.step()
        running_loss += loss.item()
    scheduler.step()
    y_true, y_score = evaluate_model(test_loader)
    auc = roc_auc_score(y_true, y_score)
    print(f"Epoch {epoch} (phase {phase}): loss={running_loss/len(train_loader):.4f}, test AUC={auc:.4f}")
    history.append({'epoch': epoch, 'phase': phase, 'val_auc': auc})
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), '/kaggle/working/best_efficientnet_b0.pt')
        print(f"  -> saved new best checkpoint (AUC={auc:.4f})")

Device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 138MB/s] 


Epoch 1 (phase 1): loss=0.5627, test AUC=0.5909
  -> saved new best checkpoint (AUC=0.5909)
Epoch 2 (phase 1): loss=0.5433, test AUC=0.5911
  -> saved new best checkpoint (AUC=0.5911)
Epoch 3 (phase 1): loss=0.5366, test AUC=0.5893
Epoch 4 (phase 2): loss=0.4843, test AUC=0.6778
  -> saved new best checkpoint (AUC=0.6778)
Epoch 5 (phase 2): loss=0.3621, test AUC=0.7593
  -> saved new best checkpoint (AUC=0.7593)
Epoch 6 (phase 2): loss=0.2860, test AUC=0.7758
  -> saved new best checkpoint (AUC=0.7758)
Epoch 7 (phase 2): loss=0.2501, test AUC=0.7851
  -> saved new best checkpoint (AUC=0.7851)
Epoch 8 (phase 2): loss=0.2304, test AUC=0.7932
  -> saved new best checkpoint (AUC=0.7932)
Epoch 9 (phase 2): loss=0.2216, test AUC=0.7947
  -> saved new best checkpoint (AUC=0.7947)
Epoch 10 (phase 2): loss=0.2103, test AUC=0.8000
  -> saved new best checkpoint (AUC=0.8000)


In [20]:
model.load_state_dict(torch.load('/kaggle/working/best_efficientnet_b0.pt'))
y_true, y_score = evaluate_model(test_loader)
y_pred = (y_score > 0.5).astype(int)
effnet_metrics = compute_metrics(y_true, y_pred, y_score)

import pandas as pd
comparison = pd.DataFrame([
    {'Detector': 'FFT Frequency Detector', **fft_metrics},
    {'Detector': 'EfficientNet-B0 (fine-tuned)', **effnet_metrics},
])
print(comparison)
comparison.to_csv('/kaggle/working/benchmark_results_official_split.csv', index=False)

                       Detector  accuracy  precision    recall        f1  \
0        FFT Frequency Detector  0.758929   0.758929  1.000000  0.862944   
1  EfficientNet-B0 (fine-tuned)  0.809263   0.858774  0.896029  0.877006   

    auc_roc  
0  0.492490  
1  0.799958  


In [1]:
yt_real_videos = [n for n in names if n.startswith('YouTube-real/') and n.endswith('.mp4')]
all_test_candidates = [(v,'real') for v in real_videos] + [(v,'fake') for v in fake_videos] + [(v,'real') for v in yt_real_videos]
test_pool_full = [(v,l) for v,l in all_test_candidates if v in test_set]
print(f"Full official test pool: {len(test_pool_full)} videos (should be 518)")

existing_test_files = set(os.listdir(f'{OUT_DIR}/test/real')) | set(os.listdir(f'{OUT_DIR}/test/fake'))
def already_extracted(entry):
    safe_name = entry[0].replace('/', '__').replace('.mp4', '')
    return any(f.startswith(safe_name) for f in existing_test_files)

missing_test = [e for e in test_pool_full if not already_extracted(e)]
print(f"Missing, will extract now: {len(missing_test)}")

for i, entry in enumerate(missing_test):
    extract_frames(entry, FRAMES_PER_VIDEO_TEST, 'test')
print("Gap filled.")

# Rebuild test datasets so they include the new frames
test_ds_model = CelebDFFrameDataset('/kaggle/working/frames', 'test', model_transform)
test_loader = DataLoader(test_ds_model, batch_size=32, shuffle=False, num_workers=2)

NameError: name 'names' is not defined

In [2]:
import os
print("Zip exists:", os.path.exists('/kaggle/working/Celeb-DF-v2.zip'))
print("Frames dir:", os.listdir('/kaggle/working/frames') if os.path.exists('/kaggle/working/frames') else "MISSING")
print("Checkpoints:", [f for f in os.listdir('/kaggle/working') if f.endswith('.pt')])

Zip exists: False
Frames dir: MISSING
Checkpoints: []


In [3]:
!pip install gdown
!gdown --fuzzy https://drive.google.com/open?id=1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj -O /kaggle/working/Celeb-DF-v2.zip

Downloading...
From (original): https://drive.google.com/uc?id=1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj
From (redirected): https://drive.google.com/uc?id=1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj&confirm=t&uuid=428bce81-a529-415d-be2a-d4e8cc448b71
To: /kaggle/working/Celeb-DF-v2.zip
100%|███████████████████████████████████████| 9.95G/9.95G [01:05<00:00, 152MB/s]


In [4]:
import zipfile, cv2, numpy as np, os, random, re, tempfile, time
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

random.seed(42)
ZIP_PATH = '/kaggle/working/Celeb-DF-v2.zip'
OUT_DIR = '/kaggle/working/frames'
FRAMES_PER_VIDEO_TRAIN = 15
FRAMES_PER_VIDEO_TEST = 20

z = zipfile.ZipFile(ZIP_PATH)
names = z.namelist()
real_videos = [n for n in names if n.startswith('Celeb-real/') and n.endswith('.mp4')]
fake_videos = [n for n in names if n.startswith('Celeb-synthesis/') and n.endswith('.mp4')]

with z.open('List_of_testing_videos.txt') as f:
    test_lines = f.read().decode('utf-8').strip().split('\n')
test_set = set(line.strip().split()[1] for line in test_lines)

def extract_frames(entry, n_frames, split):
    video_path, label = entry
    safe_name = video_path.replace('/', '__').replace('.mp4', '')
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
        tmp.write(z.read(video_path)); tmp_path = tmp.name
    saved = 0
    try:
        cap = cv2.VideoCapture(tmp_path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total > 0:
            target_idxs = set(np.linspace(0, total - 1, min(n_frames, total)).astype(int).tolist())
            i = 0
            while True:
                ok, frame = cap.read()
                if not ok: break
                if i in target_idxs:
                    cv2.imwrite(f'{OUT_DIR}/{split}/{label}/{safe_name}_{saved:03d}.jpg', frame)
                    saved += 1
                    if saved >= len(target_idxs): break
                i += 1
        cap.release()
    finally:
        os.remove(tmp_path)
    return saved

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

class CelebDFFrameDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        import glob
        self.transform = transform
        real = glob.glob(os.path.join(root_dir, split, 'real', '*.jpg'))
        fake = glob.glob(os.path.join(root_dir, split, 'fake', '*.jpg'))
        self.samples = [(p, 0) for p in real] + [(p, 1) for p in fake]
        print(f"{split}: {len(real)} real, {len(fake)} fake")
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label, path

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
model_transform = T.Compose([T.Resize((224,224)), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

train_ds_model = CelebDFFrameDataset(OUT_DIR, 'train', model_transform)
test_ds_model  = CelebDFFrameDataset(OUT_DIR, 'test', model_transform)
train_loader = DataLoader(train_ds_model, batch_size=32, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_ds_model, batch_size=32, shuffle=False, num_workers=2)

print("\nSession state restored. Ready to continue.")

Device: cuda
train: 0 real, 0 fake
test: 0 real, 0 fake


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [5]:
# Make sure output folders exist
for split in ['train', 'test']:
    for label in ['real', 'fake']:
        os.makedirs(f'{OUT_DIR}/{split}/{label}', exist_ok=True)

yt_real_videos = [n for n in names if n.startswith('YouTube-real/') and n.endswith('.mp4')]

all_videos = [(v,'real') for v in real_videos] + [(v,'fake') for v in fake_videos] + [(v,'real') for v in yt_real_videos]
train_pool = [(v,l) for v,l in all_videos if v not in test_set]
test_pool  = [(v,l) for v,l in all_videos if v in test_set]
print(f"Train pool: {len(train_pool)} videos | Test pool: {len(test_pool)} videos (should be 518)")

train_real = [x for x in train_pool if x[1]=='real']; random.shuffle(train_real)
train_fake = [x for x in train_pool if x[1]=='fake']; random.shuffle(train_fake)
TRAIN_VIDEO_CAP = 2000
n_real = min(len(train_real), TRAIN_VIDEO_CAP // 3)
n_fake = min(len(train_fake), TRAIN_VIDEO_CAP - n_real)
train_selected = train_real[:n_real] + train_fake[:n_fake]
print(f"Train selected: {len(train_selected)} ({n_real} real, {n_fake} fake)")

total = 0
for i, entry in enumerate(train_selected):
    total += extract_frames(entry, FRAMES_PER_VIDEO_TRAIN, 'train')
    if i % 200 == 0: print(f"train {i}/{len(train_selected)}, frames so far: {total}")

for i, entry in enumerate(test_pool):
    total += extract_frames(entry, FRAMES_PER_VIDEO_TEST, 'test')
    if i % 100 == 0: print(f"test {i}/{len(test_pool)}, frames so far: {total}")

print(f"\nDone. Total frames: {total}")
for split in ['train', 'test']:
    for label in ['real', 'fake']:
        print(split, label, len(os.listdir(f'{OUT_DIR}/{split}/{label}')))

Train pool: 6011 videos | Test pool: 518 videos (should be 518)
Train selected: 2000 (666 real, 1334 fake)
train 0/2000, frames so far: 15
train 200/2000, frames so far: 3015
train 400/2000, frames so far: 6001
train 600/2000, frames so far: 9001
train 800/2000, frames so far: 12001
train 1000/2000, frames so far: 15001
train 1200/2000, frames so far: 18001
train 1400/2000, frames so far: 21001
train 1600/2000, frames so far: 24001
train 1800/2000, frames so far: 27001
test 0/518, frames so far: 30006
test 100/518, frames so far: 32006
test 200/518, frames so far: 34006
test 300/518, frames so far: 36006
test 400/518, frames so far: 38006
test 500/518, frames so far: 40006

Done. Total frames: 40346
train real 9976
train fake 20010
test real 3560
test fake 6800


In [6]:
train_ds_model = CelebDFFrameDataset(OUT_DIR, 'train', model_transform)
test_ds_model  = CelebDFFrameDataset(OUT_DIR, 'test', model_transform)
train_loader = DataLoader(train_ds_model, batch_size=32, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_ds_model, batch_size=32, shuffle=False, num_workers=2)

train: 9976 real, 20010 fake
test: 3560 real, 6800 fake


In [11]:
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    resnet50, ResNet50_Weights,
    mobilenet_v3_large, MobileNet_V3_Large_Weights,
)

def build_efficientnet_b0():
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    head = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, 1))
    m.classifier = head
    return m, head

def build_resnet50():
    m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    head = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, 1))
    m.fc = head
    return m, head

def build_mobilenet_v3_large():
    m = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1)
    in_features = m.classifier[0].in_features
    head = nn.Sequential(nn.Linear(in_features, 256), nn.Hardswish(),
                          nn.Dropout(0.3), nn.Linear(256, 1))
    m.classifier = head
    return m, head

MODEL_REGISTRY = {
    'efficientnet_b0': build_efficientnet_b0,
    'resnet50': build_resnet50,
    'mobilenet_v3_large': build_mobilenet_v3_large,
}

def prepare_model(model_key):
    model, head_module = MODEL_REGISTRY[model_key]()
    model = model.to(device)
    head_ids = set(id(p) for p in head_module.parameters())
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    head_params = list(head_module.parameters())
    return model, backbone_params, head_params

def evaluate(model):
    model.eval(); labels_all, scores_all = [], []
    with torch.no_grad():
        for imgs, labels, _ in test_loader:
            probs = torch.sigmoid(model(imgs.to(device)).squeeze(1)).cpu().numpy()
            scores_all.extend(probs); labels_all.extend(labels.numpy())
    return np.array(labels_all), np.array(scores_all)

def train_and_evaluate(model_key, epochs_p1=3, epochs_p2=7, seed=42):
    torch.manual_seed(seed)
    model, backbone_params, head_params = prepare_model(model_key)
    for p in backbone_params: p.requires_grad = False
    optimizer = torch.optim.AdamW(head_params, lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs_p1)
    criterion = nn.BCEWithLogitsLoss()
    best_auc, history = -1, []

    for epoch in range(1, epochs_p1 + epochs_p2 + 1):
        if epoch == epochs_p1 + 1:
            for p in backbone_params: p.requires_grad = True
            optimizer = torch.optim.AdamW(
                [{'params': head_params, 'lr': 1e-4},
                 {'params': backbone_params, 'lr': 1e-5}], weight_decay=1e-4)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs_p2)

        model.train(); running_loss = 0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(device), labels.float().to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs).squeeze(1), labels)
            loss.backward(); optimizer.step()
            running_loss += loss.item()
        scheduler.step()

        y_true, y_score = evaluate(model)
        auc = roc_auc_score(y_true, y_score)
        print(f"[{model_key}] epoch {epoch}: loss={running_loss/len(train_loader):.4f}, AUC={auc:.4f}")
        history.append({'model': model_key, 'epoch': epoch, 'val_auc': auc})
        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), f'/kaggle/working/best_{model_key}_seed{seed}.pt')

    y_true, y_score = evaluate(model)
    y_pred = (y_score > 0.5).astype(int)
    final_metrics = {
        'model': model_key, 'seed': seed,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'auc_roc': best_auc,
    }
    return history, final_metrics

In [12]:
all_history, all_final_metrics = [], []

history, metrics = train_and_evaluate('efficientnet_b0')
metrics['train_time_min'] = None  # add timing if you want it
all_history.extend(history)
all_final_metrics.append(metrics)
print(f"\nEfficientNet-B0 done: AUC={metrics['auc_roc']:.4f}")
torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 137MB/s]


[efficientnet_b0] epoch 1: loss=0.6245, AUC=0.6770
[efficientnet_b0] epoch 2: loss=0.5975, AUC=0.6969
[efficientnet_b0] epoch 3: loss=0.5881, AUC=0.6965
[efficientnet_b0] epoch 4: loss=0.4963, AUC=0.7916
[efficientnet_b0] epoch 5: loss=0.3434, AUC=0.8279
[efficientnet_b0] epoch 6: loss=0.2762, AUC=0.8389
[efficientnet_b0] epoch 7: loss=0.2398, AUC=0.8458
[efficientnet_b0] epoch 8: loss=0.2182, AUC=0.8523
[efficientnet_b0] epoch 9: loss=0.2065, AUC=0.8520
[efficientnet_b0] epoch 10: loss=0.1989, AUC=0.8547

EfficientNet-B0 done: AUC=0.8547
